# ToolGuard benchmark: analysis notebook

This notebook inspects the published CSV outputs. The default results come from a synthetic scripted simulator, not an LLM benchmark or real customer workflow. Do not merge optional Ollama results into this table.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find files whether the project folder is local or uploaded as a Kaggle dataset.
candidates = list(Path('/kaggle/input').rglob('summary.csv')) if Path('/kaggle/input').exists() else []
if not candidates:
    candidates = [Path('outputs/summary.csv')]
summary_path = next((p for p in candidates if p.exists()), None)
if summary_path is None:
    raise FileNotFoundError('summary.csv not found. Upload outputs/summary.csv with the dataset.')
summary = pd.read_csv(summary_path)
summary_path


In [ ]:
summary.sort_values(['tool_failure_rate', 'policy']).reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for policy, group in summary.groupby('policy'):
    group = group.sort_values('tool_failure_rate')
    ax.plot(group['tool_failure_rate'], group['success_rate'], marker='o', label=policy)
ax.set_xlabel('Transient tool-failure rate')
ax.set_ylabel('Task success rate')
ax.set_ylim(0, 1.05)
ax.set_title('Task completion by tool-failure rate')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for policy, group in summary.groupby('policy'):
    group = group.sort_values('tool_failure_rate')
    ax.plot(group['tool_failure_rate'], group['mean_tool_calls'], marker='o', label=policy)
ax.set_xlabel('Transient tool-failure rate')
ax.set_ylabel('Mean tool calls per task')
ax.set_title('Tool-call overhead by tool-failure rate')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Interpretation guardrails
- The scripted policies and expected mappings are defined in the accompanying code.
- A single seed is not enough for a general conclusion; repeat across seeds and report uncertainty intervals.
- The guard is allowed to use the synthetic task's known mapping, so the results measure the designed execution-control layer rather than open-ended reasoning.
- Cite the actual versioned repository or archive you publish, and do not claim university endorsement.
